## clash 정보, plddt 정보 csv파일에 포함하기 

### 필요한 함수 

In [1]:
import numpy as np 
from Bio.PDB import PDBParser

cdr_chothia = {
    "h1": (26, 32),
    "h2": (52, 56),
    "h3": (95, 102),
    "l1": (24, 34),
    "l2": (50, 56),
    "l3": (89, 97)
}

def calculate_b_factor_mean(pdb_file, cdr_type):
    parser = PDBParser(QUIET=True)

    start_residue = cdr_chothia[cdr_type][0]
    end_residue = cdr_chothia[cdr_type][1]

    structure = parser.get_structure("structure", pdb_file)
    
    # 첫 번째 체인 추출
    first_chain = next(iter(structure[0].get_chains()))
    b_factors = []
    for residue in first_chain:
        residue_id = residue.get_id()
        if residue_id[0] == " ":  # 표준 아미노산만 고려
            residue_number = residue_id[1]
            if start_residue <= residue_number <= end_residue:
                # 각 원자의 B-factor 가져오기
                for atom in residue:
                    b_factors.append(atom.get_bfactor())

    return np.mean(b_factors)


### plddt 

In [ ]:
import pandas as pd
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from data.ab_metrics import renumber_pdb  # 또는 renumber_chain

###################################################################################################################
csv_file = '/home/psh/protein-frame-flow/inference_outputs/CDRFlow_v2.3.2.1_stage_2_wt_confidence_lr_1e-3/2025-10-26_10-47-26/epoch=42-step=61619/run_2025-11-08_22-44-09/get_capri/capri_info.csv'
base_dir = Path('/home/psh/protein-frame-flow/inference_outputs/CDRFlow_v2.3.2.1_stage_2_wt_confidence_lr_1e-3/2025-10-26_10-47-26/epoch=42-step=61619/run_2025-11-08_22-44-09')
###################################################################################################################

cdr_list = ['h1', 'h2', 'h3', 'l1', 'l2', 'l3']
df = pd.read_csv(csv_file)

for cdr in cdr_list:
    df[f"{cdr}_plddt"] = None

# ---- 병렬 처리 함수 정의 ----
def process_row(row):
    import traceback
    from data.ab_metrics import renumber_pdb
    try:
        pdb_path = str(base_dir / row['pdb_id'] / row['data_name'] / row['sample_id'] / 'get_capri' / 'sample_1_rechain.pdb')
        output_path = str(base_dir / row['pdb_id'] / row['data_name'] / row['sample_id'] / 'get_capri' / 'sample_1_chothia.pdb')

        # 1️⃣ renumber
        _ = renumber_pdb(pdb_path, out_pdb_file=output_path)

        # 2️⃣ pLDDT 계산
        cdr_plddts = {}
        for cdr in cdr_list:
            cdr_plddts[cdr] = calculate_b_factor_mean(output_path, cdr)

        return (row.name, cdr_plddts, None)

    except Exception as e:
        return (row.name, None, str(e) + "\n" + traceback.format_exc())

# ---- 병렬 실행 ----
results = []
num_workers = min(8, os.cpu_count())  # 병렬 프로세스 수 제한
print(f"🧠 Using {num_workers} parallel workers")

with ProcessPoolExecutor(max_workers=num_workers) as executor:
    futures = [executor.submit(process_row, row) for _, row in df.iterrows()]
    for i, f in enumerate(as_completed(futures), 1):
        idx, cdr_plddts, err = f.result()
        if err:
            print(f"❌ Error at index {idx}: {err}")
            continue
        for cdr in cdr_list:
            df.at[idx, f"{cdr}_plddt"] = cdr_plddts[cdr]
        if i % 10 == 0 or i == len(df):
            print(f"✅ Progress: {i}/{len(df)} done")

# ---- CSV 저장 ----
output_csv = csv_file.replace(".csv", "_with_plddt.csv")
df.to_csv(output_csv, index=False)
print(f"\n✅ Saved updated CSV to: {output_csv}")


### plddt 계산 (w/o dockq 계산)

In [5]:
import pandas as pd
import os
from concurrent.futures import ProcessPoolExecutor, as_completed
from data.ab_metrics import renumber_pdb_wt_constant
from experiments.utils import calculate_b_factor_mean

###################################################################################################################
base_dir = '/home/psh/protein-frame-flow/inference_outputs/CDRFlow_v2.3.2.1_stage_2_wt_confidence_lr_1e-3/2025-10-26_10-47-26/epoch=42-step=61619/run_2025-11-09_16-59-39'
###################################################################################################################

cdr_list = ['h1', 'h2', 'h3', 'l1', 'l2', 'l3']
os.makedirs(os.path.join(base_dir, 'get_capri'), exist_ok=True)
csv_file = os.path.join(base_dir, 'get_capri', 'capri_info_with_plddt.csv')


def process_sample(args):
    """단일 샘플 처리 함수 (병렬 실행용)"""
    pdb_id, kkh_sample, sample, sample_path, output_path = args

    if not os.path.exists(sample_path):
        return None

    try:
        _ = renumber_pdb_wt_constant(sample_path, output_path)
    except Exception as e:
        print(f"[WARN] renumber_pdb_wt_constant failed for {sample_path}: {e}")
        return None

    cdr_plddts = {}
    for cdr in cdr_list:
        try:
            cdr_plddts[cdr] = calculate_b_factor_mean(output_path, cdr)
        except Exception as e:
            print(f"[WARN] Failed to calculate {cdr} for {output_path}: {e}")
            cdr_plddts[cdr] = None

    return {
        'pdb_id': pdb_id,
        'data_name': kkh_sample,
        'sample_id': sample,
        **{f'{cdr}_plddt': cdr_plddts[cdr] for cdr in cdr_list}
    }


# ================================================================================================================
# 1️⃣ 처리할 모든 샘플 경로를 미리 수집
# ================================================================================================================
tasks = []
for pdb_id in os.listdir(base_dir):
    if 'config' in pdb_id:
        continue
    pdb_dir = os.path.join(base_dir, pdb_id)

    for kkh_sample in os.listdir(pdb_dir):
        if 'capri' in kkh_sample:
            continue
        kkh_sample_dir = os.path.join(pdb_dir, kkh_sample)

        for sample in os.listdir(kkh_sample_dir):
            sample_path = os.path.join(kkh_sample_dir, sample, 'sample_1.pdb')
            output_path = os.path.join(kkh_sample_dir, sample, 'sample_1_chothia.pdb')
            tasks.append((pdb_id, kkh_sample, sample, sample_path, output_path))

print(f"[INFO] Total {len(tasks)} samples found. Starting multiprocessing...")

# ================================================================================================================
# 2️⃣ 멀티프로세싱 실행
# ================================================================================================================
records = []
max_workers = 20  # 모든 CPU 코어 사용
with ProcessPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(process_sample, task) for task in tasks]
    for i, future in enumerate(as_completed(futures), start=1):
        result = future.result()
        if result is not None:
            records.append(result)
        if i % 50 == 0:
            print(f"[INFO] Processed {i}/{len(futures)} samples")

# ================================================================================================================
# 3️⃣ 결과 CSV로 저장
# ================================================================================================================
df = pd.DataFrame(records, columns=['pdb_id', 'data_name', 'sample_id'] + [f'{cdr}_plddt' for cdr in cdr_list])
os.makedirs(os.path.dirname(csv_file), exist_ok=True)
df.to_csv(csv_file, index=False)
print(f"[INFO] CSV saved to {csv_file} ({len(df)} rows)")


chain_res [<Residue GLN het=  resseq=1 icode= >, <Residue VAL het=  resseq=2 icode= >, <Residue GLN het=  resseq=3 icode= >, <Residue LEU het=  resseq=4 icode= >, <Residue VAL het=  resseq=5 icode= >, <Residue GLN het=  resseq=6 icode= >, <Residue SER het=  resseq=7 icode= >, <Residue GLY het=  resseq=8 icode= >, <Residue ALA het=  resseq=9 icode= >, <Residue GLU het=  resseq=10 icode= >, <Residue VAL het=  resseq=11 icode= >, <Residue LYS het=  resseq=12 icode= >, <Residue LYS het=  resseq=13 icode= >, <Residue PRO het=  resseq=14 icode= >, <Residue GLY het=  resseq=15 icode= >, <Residue ALA het=  resseq=16 icode= >, <Residue SER het=  resseq=17 icode= >, <Residue VAL het=  resseq=18 icode= >, <Residue LYS het=  resseq=19 icode= >, <Residue VAL het=  resseq=20 icode= >, <Residue SER het=  resseq=21 icode= >, <Residue CYS het=  resseq=22 icode= >, <Residue LYS het=  resseq=23 icode= >, <Residue ALA het=  resseq=24 icode= >, <Residue SER het=  resseq=25 icode= >, <Residue GLY het=  ress

## rmsd 계산 

In [ ]:
from Bio import PDB
import numpy as np
import os
import pandas as pd 

########################################################################################################
kkh_dir = "/home/psh/protein-frame-flow/inference_outputs/kkh_20251020_chothia"
cdrflow_dir = '/home/psh/protein-frame-flow/inference_outputs/CDRFlow_v2.3.2.1_stage_2_wt_confidence_lr_1e-3/2025-10-26_10-47-26/epoch=42-step=61619/kkh'
plddt_csv = '/home/psh/protein-frame-flow/inference_outputs/CDRFlow_v2.3.2.1_stage_2_wt_confidence_lr_1e-3/2025-10-26_10-47-26/epoch=42-step=61619/kkh/get_capri/capri_info_with_plddt_merged_with_kkh.csv'
########################################################################################################
csv_info = pd.read_csv(plddt_csv)

# -------------------------------
# Chothia numbering 기준 CDR residue ranges
# -------------------------------
CDR_DEFS = {
    "h1": (26, 32),
    "h2": (52, 56),
    "h3": (95, 102),
    "l1": (24, 34),
    "l2": (50, 56),
    "l3": (89, 97),
}

# -------------------------------
# Chain ID 기준 Heavy/Light 순서 지정
# -------------------------------
def get_chain_sequences_by_id(structure):
    chains = list(structure.get_chains())
    heavy_chains = [c for c in chains if c.id.upper() == 'A']
    light_chains = [c for c in chains if c.id.upper() == 'B']
    if not heavy_chains or not light_chains:
        heavy_chains = [chains[0]]
        light_chains = [chains[1]]
    return heavy_chains + light_chains

# -------------------------------
# Residue.id[1] 기준 dict 생성
# -------------------------------
def get_residues_dict(chain):
    return {f"{res.id[1]}_{res.id[2]}": res for res in chain if PDB.is_aa(res, standard=True)}

# -------------------------------
# 공통 backbone atom 추출 (heavy+light chain)
# -------------------------------
def get_common_atoms_all(chains_ref, chains_mob, only_frame=False):
    ref_atoms_all = []
    mob_atoms_all = []
    common_residue_map = {}  # chain_idx -> res_id -> (ref_atom_indices, mob_atom_indices)

    for chain_idx, (chain_ref, chain_mob) in enumerate(zip(chains_ref, chains_mob)):
        residues_ref = get_residues_dict(chain_ref)
        residues_mob = get_residues_dict(chain_mob)
        common_res_ids = sorted(set(residues_ref.keys()) & set(residues_mob.keys()))

        chain_ref_indices = []
        chain_mob_indices = []

        # CDR 범위 정의
        if chain_idx == 0:
            cdr_ranges = [CDR_DEFS[k] for k in CDR_DEFS if k.startswith('h')]
        else:
            cdr_ranges = [CDR_DEFS[k] for k in CDR_DEFS if k.startswith('l')]

        for rid in common_res_ids:
            # only_frame=True면 CDR 영역 제외
            res_num = int(rid.split('_')[0])
            if only_frame and any(start <= res_num <= end for start, end in cdr_ranges):
                continue  # skip CDR residues if only_frame

            res_ref = residues_ref[rid]
            res_mob = residues_mob[rid]
            for atom_name in ["N", "CA", "C"]:
                if atom_name in res_ref and atom_name in res_mob:
                    ref_atoms_all.append(res_ref[atom_name])
                    mob_atoms_all.append(res_mob[atom_name])
                    chain_ref_indices.append(len(ref_atoms_all)-1)
                    chain_mob_indices.append(len(mob_atoms_all)-1)

            # chain_idx -> res_id -> (ref_indices, mob_indices)
            if chain_ref_indices and chain_mob_indices:
                common_residue_map.setdefault(chain_idx, {})[rid] = (
                    chain_ref_indices[-3:], chain_mob_indices[-3:]
                )

    return ref_atoms_all, mob_atoms_all, common_residue_map

# -------------------------------
# Superpose 구조
# -------------------------------
def superpose_structures(ref_atoms, mob_atoms, mob_structure):
    assert len(ref_atoms) == len(mob_atoms)

    sup = PDB.Superimposer()
    sup.set_atoms(ref_atoms, mob_atoms)
    sup.apply(mob_structure.get_atoms())
    return sup.rms

# -------------------------------
# 전체 계산 함수
# -------------------------------
def compute_cdr_rmsd_save_aligned(ref_pdb, mob_pdb):
    parser = PDB.PDBParser(QUIET=True)
    s_ref = parser.get_structure("ref", ref_pdb)
    s_mob = parser.get_structure("mob", mob_pdb)

    chains_ref = get_chain_sequences_by_id(s_ref)
    chains_mob = get_chain_sequences_by_id(s_mob)

    # -------------------------------
    # 공통 backbone atom 추출 (Heavy+Light)
    # -------------------------------
    ref_atoms_all, mob_atoms_all, common_residue_map = get_common_atoms_all(chains_ref, chains_mob, only_frame=False)
    ref_atoms_frame, mob_atoms_frame, common_residue_map_frame = get_common_atoms_all(chains_ref, chains_mob, only_frame=True)

    # Superpose
    framework_rms = superpose_structures(ref_atoms_frame, mob_atoms_frame, s_mob)

    # -------------------------------
    # CDR RMSD 계산
    # -------------------------------
    all_cdr_rmsds = {}
    for chain_idx, chain in enumerate(chains_ref):
        cdr_types = [k for k in CDR_DEFS if (k.startswith('h') if chain_idx==0 else k.startswith('l'))]
        for cdr_name in cdr_types:
            start, end = CDR_DEFS[cdr_name]
            residues_map = common_residue_map.get(chain_idx, {})

            # CDR 범위 안에 들어있는 residue만 선택
            cdr_ref_indices = []
            cdr_mob_indices = []
            for rid in residues_map:
                if start <= int(rid.split('_')[0]) <= end:
                    ref_idx, mob_idx = residues_map[rid]
                    cdr_ref_indices.extend(ref_idx)
                    cdr_mob_indices.extend(mob_idx)

            diffs = np.array([ref_atoms_all[i].coord - mob_atoms_all[j].coord
                              for i,j in zip(cdr_ref_indices, cdr_mob_indices)])
            rmsd = np.sqrt(np.mean(np.sum(diffs**2, axis=1)))
            all_cdr_rmsds[cdr_name] = rmsd

    return framework_rms, all_cdr_rmsds



# ==========================================================
# 사용 예시
# ==========================================================
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm  # 진행률 표시


# ==========================================================
# RMSD 계산용 함수 (행 단위)
# ==========================================================
def process_rmsd_row(row):
    pdb_id = row['pdb_id']
    data_name = row['data_name']
    sample_id = row['sample_id']

    pdb1 = os.path.join(kkh_dir, pdb_id, data_name + '.pdb')
    pdb2 = os.path.join(cdrflow_dir, pdb_id, data_name, sample_id, 'sample_1_chothia.pdb')

    # PDB 존재 확인
    frame_rms, cdr_rmsds = compute_cdr_rmsd_save_aligned(pdb1, pdb2)


    # 결과 딕셔너리 반환
    result = {
        'frame_rmsd': frame_rms,
        'h1_rmsd': cdr_rmsds.get('h1'),
        'h2_rmsd': cdr_rmsds.get('h2'),
        'h3_rmsd': cdr_rmsds.get('h3'),
        'l1_rmsd': cdr_rmsds.get('l1'),
        'l2_rmsd': cdr_rmsds.get('l2'),
        'l3_rmsd': cdr_rmsds.get('l3'),
    }
    return row.name, result

# ==========================================================
# 병렬 처리 + 진행률 표시
# ==========================================================
num_workers = 80  # CPU 코어 수
results = {}

with ProcessPoolExecutor(max_workers=num_workers) as executor:
    futures = {executor.submit(process_rmsd_row, row): idx for idx, row in csv_info.iterrows()}
    
    # tqdm으로 진행률 표시
    for future in tqdm(as_completed(futures), total=len(futures), desc="Computing RMSDs"):
        idx, res_dict = future.result()
        results[idx] = res_dict

# ==========================================================
# 결과 CSV에 적용
# ==========================================================
for idx, res_dict in results.items():
    for col, val in res_dict.items():
        csv_info.at[idx, col] = val

# 저장
output_csv = os.path.join(cdrflow_dir, 'get_capri', 'capri_info_with_plddt_rmsd.csv')
csv_info.to_csv(output_csv, index=False)
print(f"병렬 RMSD 계산 완료. 저장된 CSV: {output_csv}")

Total common backbone atoms: 786 786
Framework RMSD: 0.252 Å
Aligned mobile structure saved to: /home/psh/protein-frame-flow/notebook/align.pdb
Framework alignment RMSD: 0.252 Å
H1 RMSD: 0.237 Å
H2 RMSD: 0.612 Å
H3 RMSD: 1.448 Å
L1 RMSD: 0.362 Å
L2 RMSD: 0.280 Å
L3 RMSD: 0.327 Å


## 기존 csv파일에 붙이기 

In [12]:
import os 
import json 
import pandas as pd 

# 파일 경로
file_a = '/home/kkh517/run-epi-a817668/results/run_20251020/raw_results.csv'
file_b = '/home/psh/protein-frame-flow/inference_outputs/CDRFlow_v2.3.2.1_stage_2_wt_confidence_lr_1e-3/2025-10-26_10-47-26/epoch=42-step=61619/run_2025-11-09_16-59-18/get_capri/capri_info_with_plddt_rmsd.csv'

# CSV 읽기
df_a = pd.read_csv(file_a)
df_b = pd.read_csv(file_b)

# 열 이름 변경
df_a = df_a.rename(columns={
    'capri_criteria': 'capri_kkh',
    'dockq_score': 'dockq_score_kkh'
})

# 필요한 열만 선택
df_a = df_a[['pdb_id', 'data_name', 'capri_kkh', 'dockq_score_kkh', 'interface_pae']]

# 병합 (inner join: 일치하는 행만)
merged = pd.merge(df_b, df_a, on=['pdb_id', 'data_name'], how='left')

# 결과 저장
output_path = file_b.replace('.csv', '_merged_with_kkh.csv')
merged.to_csv(output_path, index=False)

print(f"✅ Merged CSV saved to: {output_path}")
print(f"✅ Shape: {merged.shape}")



✅ Merged CSV saved to: /home/psh/protein-frame-flow/inference_outputs/CDRFlow_v2.3.2.1_stage_2_wt_confidence_lr_1e-3/2025-10-26_10-47-26/epoch=42-step=61619/run_2025-11-09_16-59-18/get_capri/capri_info_with_plddt_rmsd_merged_with_kkh.csv
✅ Shape: (28575, 19)


## visualization 

In [ ]:
import pandas as pd

csv_file = '/home/psh/protein-frame-flow/inference_outputs/CDRFlow_v2.3.2.1_stage_2_wt_confidence_lr_1e-3/2025-10-26_10-47-26/epoch=42-step=61619/kkh/get_capri/combined_capri_rmsd.csv'

